## Candidate case attributes

**Information Need:** Identify attributes that are likely to represent  log-level or case-level properties.

**Motivation:** Event logs may not explicitly or reliably distinguish log-level and case-level from event-level attributes. Identifying attributes that remain constant within the log and cases helps analysts understand the log structure and determine which attributes characterize the entire log or entire process instances and therefore must not be analyzed on the event level.

**Approach:** Identify evidence that an attribute might represent log-level or case-level information, considering explicit log- and case-level designation, and whether its value remains constant within the log or within each case.

**Output:** The set of attributes identified as candidate log and case attributes, respectively.

In [ ]:
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

### 1. Case Attribute Candidates

In [ ]:
columns_to_check = [c for c in event_log.columns if c != CASE_ID]

starts_with_case_prefix = [c for c in columns_to_check if c.startswith('case:')]

max_distinct_per_case = event_log.groupby(CASE_ID)[columns_to_check].nunique().max()
unique_per_case = max_distinct_per_case[max_distinct_per_case <= 1].index.tolist()

candidate_case_attributes = sorted(set(starts_with_case_prefix) | set(unique_per_case))

print(f'Candidate case attributes ({len(candidate_case_attributes)}):')
for attribute in candidate_case_attributes:
    signals = []
    if attribute in starts_with_case_prefix:
        signals.append("case: prefix")
    if attribute in unique_per_case:
        signals.append("unique per case")
    print(f"- {attribute} ({', '.join(signals)})")

### 2. Log Attribute Candidates

In [ ]:
starts_with_log_prefix = [c for c in columns_to_check if c.startswith('log:')]

distinct_overall = event_log[columns_to_check].nunique()
unique_overall = distinct_overall[distinct_overall <= 1].index.tolist()

candidate_log_attributes = sorted(set(starts_with_log_prefix) | set(unique_overall))

print(f'Candidate log attributes ({len(candidate_log_attributes)}):')
for attribute in candidate_log_attributes:
    signals = []
    if attribute in starts_with_log_prefix:
        signals.append("log: prefix")
    if attribute in unique_overall:
        signals.append("unique across log")
    print(f"- {attribute} ({', '.join(signals)})")